# XLM-RoBERTa Sentiment Pipeline (Shopee Taglish)

Fine-tune `xlm-roberta-base` on ~1,000 manually labeled Shopee reviews
(negative / neutral / positive), then score the unlabeled pool and rank
products with a Bayesian Weighted Score.

**Runtime:** Colab GPU (T4).

**If training already crashed on `warmup_ratio`:** skip the install and
data cells. Re-run **Helper functions**, then **Train**. The backbone
weights are already cached — you do not need to download them again.

The `UNEXPECTED` / `MISSING` load report for `lm_head` vs `classifier` is
normal when turning a masked-language-model checkpoint into a 3-class
classifier. Ignore it.


## 1. Install


In [ ]:
# Compatible with transformers 4.x and 5.x (Colab currently ships 5.x).
# Do NOT pin <5.0 — that would force a runtime restart after the model
# has already downloaded.
%pip -q install "transformers>=4.40.0" datasets accelerate scikit-learn sentencepiece evaluate


## 2. Drive + config


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import os
import random
import re
import inspect
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.nn import functional as F

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

# ---------------------------------------------------------------------------
# Paths — edit DRIVE_ROOT if your folder name differs
# ---------------------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/sentiment-analysis-system"
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "outputs")
MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "xlmr_sentiment_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Training recipe
# ---------------------------------------------------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
LABEL_COL = "Final_Label"
TEXT_COL = "Comment"
RATING_COL = "Rating Star"

SEED = 42
TEST_SIZE = 0.15
EPOCHS = 6
LR = 1.5e-5
BATCH_SIZE = 8
GRAD_ACCUM = 2
MAX_LENGTH = 192
WARMUP_RATIO = 0.10
DROPOUT = 0.20
LABEL_SMOOTHING = 0.08
FOCAL_GAMMA = 1.5
NEUTRAL_WEIGHT_BOOST = 1.25

USE_RATING_CONTEXT = True
USE_SELF_TRAINING = True
SELF_TRAIN_CONF = 0.92
SELF_TRAIN_MAX = 3500

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("transformers:", __import__("transformers").__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("DRIVE_ROOT:", DRIVE_ROOT)


## 3. Load labeled data + split


In [ ]:
def _first_existing(paths):
    for path in paths:
        if path and os.path.isfile(path):
            return path
    return None


labeled_path = _first_existing(
    [
        os.path.join(DATA_DIR, "manual_labeled_data.csv"),
        os.path.join(DATA_DIR, "labeled.csv"),
        os.path.join(DRIVE_ROOT, "manual_labeled_data.csv"),
        "/content/manual_labeled_data.csv",
    ]
)
if labeled_path is None:
    # Fall back: any CSV in data/ whose columns look labeled
    candidates = []
    search_dirs = [DATA_DIR, DRIVE_ROOT, "/content"]
    for folder in search_dirs:
        if not os.path.isdir(folder):
            continue
        for name in os.listdir(folder):
            if name.lower().endswith(".csv"):
                candidates.append(os.path.join(folder, name))
    for path in candidates:
        try:
            peek = pd.read_csv(path, nrows=3)
        except Exception:
            continue
        cols = {c.lower() for c in peek.columns}
        if "final_label" in cols or (("comment" in cols or "review" in cols) and "label" in cols):
            labeled_path = path
            break

if labeled_path is None:
    raise FileNotFoundError(
        "Could not find the labeled CSV. Place manual_labeled_data.csv under "
        f"{DATA_DIR} or update labeled_path."
    )

df = pd.read_csv(labeled_path)
# Normalize expected column names
rename = {}
for col in df.columns:
    key = col.strip().lower()
    if key in {"final_label", "label", "sentiment"} and LABEL_COL not in df.columns:
        rename[col] = LABEL_COL
    elif key in {"comment", "review", "review_text", "text", "product_review_text"} and TEXT_COL not in df.columns:
        rename[col] = TEXT_COL
    elif key in {"rating star", "rating", "stars", "star"} and RATING_COL not in df.columns:
        rename[col] = RATING_COL
if rename:
    df = df.rename(columns=rename)

df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
df = df[df[LABEL_COL].isin(LABEL2ID)].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
print("Labeled file:", labeled_path)
print("Labeled rows:", len(df))
print("Label distribution:")
print(df[LABEL_COL].value_counts())
if RATING_COL in df.columns:
    print("\nRating vs label:")
    print(pd.crosstab(df[RATING_COL], df[LABEL_COL]))

train_df, val_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[LABEL_COL],
)
print(f"\nTrain: {len(train_df)}  |  Val: {len(val_df)}")
split_path = os.path.join(OUTPUT_DIR, "train_test_split.csv")
split_out = df.copy()
split_out["split"] = "train"
split_out.loc[val_df.index, "split"] = "val"
split_out.to_csv(split_path, index=False)
print("Held-out split saved →", split_path)


## 4. Helper functions (transformers 4.x **and** 5.x)

Colab's transformers 5 dropped `warmup_ratio` and `evaluation_strategy`.
The old try/except swapped only the eval-strategy name and still passed
`warmup_ratio`, which is why both branches crashed.

These helpers inspect the constructor and only pass supported kwargs.


In [ ]:
# ---- text formatting (must match the Streamlit app) ----
SHOPEE_FIELD_RE = re.compile(
    r"(?im)^\s*(Product Quality|Performance|Best Feature|Worst Feature|Others?)\s*:\s*"
)


def clean_review(text) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    body = SHOPEE_FIELD_RE.sub("", str(text))
    return re.sub(r"\s+", " ", body).strip()


def build_model_text(text, rating=None) -> str:
    body = clean_review(text)
    if not USE_RATING_CONTEXT:
        return body
    try:
        if rating is None or (isinstance(rating, float) and np.isnan(rating)):
            return body
        stars = int(float(rating))
        if stars < 1 or stars > 5:
            return body
    except (TypeError, ValueError):
        return body
    return f"Star rating: {stars} out of 5.\nReview: {body}"


def rating_of(row):
    if RATING_COL in getattr(row, "index", []):
        return row[RATING_COL]
    return None


# ---- transformers v4/v5 TrainingArguments ----
TYPEERROR_KWARG_RE = re.compile(r"unexpected keyword argument ['\"](\w+)['\"]")
TRAINING_ARG_ALIASES = (
    ("eval_strategy", "evaluation_strategy"),
    ("warmup_ratio", "warmup_steps"),
)
TRAINER_ALIASES = (("processing_class", "tokenizer"),)


def _param_names(init_fn):
    params = inspect.signature(init_fn).parameters
    has_var = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
    names = {n for n in params if n != "self"}
    return names, has_var


def filter_init_kwargs(init_fn, kwargs, exclusive_groups=()):
    names, has_var = _param_names(init_fn)
    selected = dict(kwargs)
    for group in exclusive_groups:
        present = [k for k in group if k in selected]
        if len(present) <= 1:
            continue
        chosen = next((k for k in group if has_var or k in names), None)
        if chosen is None:
            for k in present:
                selected.pop(k, None)
        else:
            for k in present:
                if k != chosen:
                    selected.pop(k, None)
    if has_var:
        return selected
    return {k: v for k, v in selected.items() if k in names}


def instantiate_filtering(cls, kwargs, exclusive_groups=()):
    filtered = filter_init_kwargs(cls.__init__, kwargs, exclusive_groups)
    while True:
        try:
            return cls(**filtered)
        except TypeError as exc:
            match = TYPEERROR_KWARG_RE.search(str(exc))
            if not match:
                raise
            bad = match.group(1)
            if bad not in filtered:
                raise
            filtered.pop(bad)


def make_training_args(output_dir, epochs):
    kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=20,
        save_strategy="epoch",
        eval_strategy="epoch",
        evaluation_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        seed=SEED,
        report_to="none",
        fp16=torch.cuda.is_available(),
        warmup_ratio=WARMUP_RATIO,
        warmup_steps=WARMUP_RATIO,  # v5: float in [0, 1) = ratio of total steps
        overwrite_output_dir=True,
        dataloader_num_workers=0,
    )
    args = instantiate_filtering(TrainingArguments, kwargs, TRAINING_ARG_ALIASES)
    used = filter_init_kwargs(TrainingArguments.__init__, kwargs, TRAINING_ARG_ALIASES)
    print(
        "eval key:",
        "eval_strategy" if "eval_strategy" in used else "evaluation_strategy",
        "| warmup key:",
        "warmup_ratio" if "warmup_ratio" in used else "warmup_steps",
        "=",
        used.get("warmup_ratio", used.get("warmup_steps")),
    )
    return args


class FocalTrainer(Trainer):
    def __init__(self, *args, class_weights=None, focal_gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights
        self._focal_gamma = float(focal_gamma)
        self._label_smoothing = float(label_smoothing)
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self._class_weights
        if weights is not None:
            weights = weights.to(device=logits.device, dtype=logits.dtype)
        per_item = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            weight=weights,
            reduction="none",
            label_smoothing=self._label_smoothing,
        )
        pt = torch.exp(-per_item)
        loss = (((1.0 - pt) ** self._focal_gamma) * per_item).mean()
        return (loss, outputs) if return_outputs else loss


def make_class_weights(labels):
    counts = Counter(labels)
    n = max(sum(counts.values()), 1)
    k = len(ID2LABEL)
    weights = []
    for i, name in ID2LABEL.items():
        c = max(counts.get(name, 1), 1)
        w = n / (k * c)
        if name == "neutral":
            w *= NEUTRAL_WEIGHT_BOOST
        weights.append(w)
    tensor = torch.tensor(weights, dtype=torch.float)
    pretty = {ID2LABEL[i]: round(float(weights[i]), 3) for i in range(k)}
    print("Class weights:", pretty)
    return tensor


def oversample_neutral(frame, target_count=None):
    parts = []
    counts = frame[LABEL_COL].value_counts()
    target = int(target_count or counts.max())
    rng = np.random.default_rng(SEED)
    for label, group in frame.groupby(LABEL_COL, sort=False):
        extra = []
        if len(group) < target:
            sampled = group.sample(target - len(group), replace=True, random_state=SEED)
            if label == "neutral":
                dropped = []
                for text in sampled[TEXT_COL].tolist():
                    words = str(text).split()
                    if len(words) > 6:
                        keep = rng.random(len(words)) > 0.12
                        keep[0] = True
                        text = " ".join(w for w, k in zip(words, keep) if k)
                    dropped.append(text)
                sampled = sampled.copy()
                sampled[TEXT_COL] = dropped
            extra.append(sampled)
        parts.append(pd.concat([group, *extra], ignore_index=True))
    out = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print("After oversampling:", len(out), "rows")
    print(out[LABEL_COL].value_counts())
    return out


def tokenize_frame(frame, tokenizer):
    texts = [
        build_model_text(row[TEXT_COL], row[RATING_COL] if RATING_COL in frame.columns else None)
        for _, row in frame.iterrows()
    ]
    labels = [LABEL2ID[str(x).strip().lower()] for x in frame[LABEL_COL].tolist()]
    ds = Dataset.from_dict({"text": texts, "label": labels})

    def _tok(batch):
        enc = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)
        enc["labels"] = batch["label"]
        return enc

    return ds.map(_tok, batched=True, remove_columns=["text", "label"])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def load_backbone():
    print("Loading model:", MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
    )
    return model


def make_trainer(model, args, train_ds, eval_ds, tokenizer, class_weights):
    kw = dict(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights,
    )
    filtered = filter_init_kwargs(Trainer.__init__, kw, TRAINER_ALIASES)
    # class_weights is a FocalTrainer extra, not a Trainer arg
    filtered["class_weights"] = class_weights
    return FocalTrainer(**filtered)


print("Helpers ready. Re-run the Train cell next.")


## 5. Train


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
val_ds = tokenize_frame(val_df, tokenizer)


def run_round(train_frame, round_name, epochs):
    print(f"\n══ {round_name} ══")
    balanced = oversample_neutral(train_frame)
    train_ds = tokenize_frame(balanced, tokenizer)
    class_weights = make_class_weights(train_frame[LABEL_COL].tolist())
    model = load_backbone()
    out_dir = os.path.join(MODEL_DIR, round_name.lower().replace(" ", "_"))
    os.makedirs(out_dir, exist_ok=True)
    args = make_training_args(out_dir, epochs)
    trainer = make_trainer(model, args, train_ds, val_ds, tokenizer, class_weights)
    trainer.train()
    metrics = trainer.evaluate()
    print(round_name, "val metrics:", {k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    return trainer, metrics


trainer, metrics = run_round(train_df, "Round 1 (gold labels)", EPOCHS)


## 6. Optional self-training

High-confidence unlabeled Shopee reviews whose predicted label agrees with
the star-rating heuristic (1–2★ negative, 3★ neutral, 4–5★ positive) are
added as extra training rows. Skip this cell if `USE_SELF_TRAINING` is False
or you do not have an unlabeled CSV.


In [ ]:
def star_heuristic(rating):
    try:
        stars = int(float(rating))
    except (TypeError, ValueError):
        return None
    if stars <= 2:
        return "negative"
    if stars == 3:
        return "neutral"
    if stars >= 4:
        return "positive"
    return None


def find_unlabeled_csv():
    names = [
        "unlabeled.csv",
        "shopee_reviews.csv",
        "reviews-final.csv",
        "reviews.csv",
    ]
    for folder in (DATA_DIR, DRIVE_ROOT, "/content"):
        if not os.path.isdir(folder):
            continue
        for name in names:
            path = os.path.join(folder, name)
            if os.path.isfile(path):
                return path
        for name in os.listdir(folder):
            if name.lower().endswith(".csv") and "label" not in name.lower():
                path = os.path.join(folder, name)
                if os.path.abspath(path) != os.path.abspath(labeled_path):
                    return path
    return None


if USE_SELF_TRAINING:
    unlabeled_path = find_unlabeled_csv()
    if unlabeled_path is None:
        print("No unlabeled CSV found — skipping self-training.")
    else:
        unlabeled = pd.read_csv(unlabeled_path)
        text_col = TEXT_COL if TEXT_COL in unlabeled.columns else next(
            (c for c in unlabeled.columns if c.lower() in {"comment", "review", "review_text", "text", "product_review_text"}),
            None,
        )
        if text_col is None:
            raise ValueError(f"No review text column in {unlabeled_path}: {list(unlabeled.columns)}")
        unlabeled = unlabeled.copy()
        unlabeled["_text"] = unlabeled[text_col].fillna("").astype(str)
        unlabeled = unlabeled[unlabeled["_text"].str.len() >= 8].copy()
        # Drop rows that are already in the gold set
        gold_texts = set(train_df[TEXT_COL].astype(str)) | set(val_df[TEXT_COL].astype(str))
        unlabeled = unlabeled[~unlabeled["_text"].isin(gold_texts)].copy()
        print("Unlabeled pool:", unlabeled_path, "rows after filters:", len(unlabeled))

        from transformers import AutoModelForSequenceClassification as _AM

        inf_model = _AM.from_pretrained(MODEL_DIR)
        inf_model.to("cuda" if torch.cuda.is_available() else "cpu")
        inf_model.eval()
        ratings = unlabeled[RATING_COL] if RATING_COL in unlabeled.columns else [None] * len(unlabeled)
        texts = [build_model_text(t, r) for t, r in zip(unlabeled["_text"], ratings)]

        all_probs = []
        bs = 32
        device = next(inf_model.parameters()).device
        with torch.no_grad():
            for i in range(0, len(texts), bs):
                enc = tokenizer(
                    texts[i : i + bs],
                    truncation=True,
                    max_length=MAX_LENGTH,
                    padding=True,
                    return_tensors="pt",
                )
                enc = {k: v.to(device) for k, v in enc.items()}
                probs = torch.softmax(inf_model(**enc).logits, dim=-1).cpu().numpy()
                all_probs.append(probs)
        probs = np.vstack(all_probs)
        pred_ids = probs.argmax(axis=1)
        conf = probs.max(axis=1)
        pred_labels = [ID2LABEL[int(i)] for i in pred_ids]

        keep_mask = []
        for i, (lab, c, rating) in enumerate(zip(pred_labels, conf, ratings)):
            if c < SELF_TRAIN_CONF:
                keep_mask.append(False)
                continue
            heuristic = star_heuristic(rating) if rating is not None and not (isinstance(rating, float) and np.isnan(rating)) else None
            keep_mask.append(heuristic is None or heuristic == lab)

        pseudo = unlabeled.loc[np.array(keep_mask)].copy()
        pseudo[LABEL_COL] = [lab for lab, k in zip(pred_labels, keep_mask) if k]
        if TEXT_COL not in pseudo.columns:
            pseudo[TEXT_COL] = pseudo["_text"]
        if len(pseudo) > SELF_TRAIN_MAX:
            pseudo = pseudo.sample(SELF_TRAIN_MAX, random_state=SEED)
        print(f"Pseudo-labeled rows kept (conf>={SELF_TRAIN_CONF}, rating-consistent): {len(pseudo)}")
        print(pseudo[LABEL_COL].value_counts())

        if len(pseudo) >= 50:
            combined = pd.concat(
                [train_df[[TEXT_COL, LABEL_COL] + ([RATING_COL] if RATING_COL in train_df.columns else [])],
                 pseudo[[TEXT_COL, LABEL_COL] + ([RATING_COL] if RATING_COL in pseudo.columns else [])]],
                ignore_index=True,
            )
            trainer, metrics = run_round(combined, "Round 2 (self-training)", max(2, EPOCHS - 2))
        else:
            print("Not enough pseudo rows — keeping Round 1 model.")
else:
    print("USE_SELF_TRAINING is False — skipping.")


## 7. Held-out validation report


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
eval_model.to(device)
eval_model.eval()

val_texts = [
    build_model_text(row[TEXT_COL], row[RATING_COL] if RATING_COL in val_df.columns else None)
    for _, row in val_df.iterrows()
]
y_true = [LABEL2ID[str(x).strip().lower()] for x in val_df[LABEL_COL].tolist()]
y_pred = []
with torch.no_grad():
    for i in range(0, len(val_texts), 32):
        enc = tokenizer(
            val_texts[i : i + 32],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        y_pred.extend(eval_model(**enc).logits.argmax(dim=-1).cpu().tolist())

print(classification_report(y_true, y_pred, target_names=[ID2LABEL[i] for i in range(3)], digits=4))
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")
print(f"Accuracy: {acc:.4f}  |  Weighted F1: {f1:.4f}")
print("Target for the thesis comparison: 0.85–0.90. Neutral is the hard class.")

metrics_path = os.path.join(OUTPUT_DIR, "val_metrics.json")
import json as _json

with open(metrics_path, "w") as f:
    _json.dump({"accuracy": acc, "weighted_f1": f1, "n_val": len(y_true)}, f, indent=2)
print("Saved", metrics_path)
print("Model saved →", MODEL_DIR)


## 8. Notes

- **HF Hub unauthenticated warning** is harmless for the public `xlm-roberta-base` download. Optional: `os.environ["HF_TOKEN"] = "hf_..."`.
- Copy `models/xlmr_sentiment_model/` next to `app.py` so Streamlit loads the same checkpoint. Inference must use the same `Star rating: N out of 5.\nReview: ...` format (`model.py` does this when a rating is provided).
- If validation is still below ~0.85, keep `USE_RATING_CONTEXT = True` (the main lever on this 1k set) and check that the val split still has mixed 3★ / 4★ neutrals.
